# Grounding Answers in Real Documents

**Notebook 4 of the main path.** Notebook 3 showed something uncomfortable:
the real model can write a fluent, specific, completely confident-sounding
number — a pressure test value — with nothing real behind it at all. This
notebook asks the natural next question: **can we fix that?**

The short answer, which we'll prove with the real model rather than just
claim, is: not by trusting the model's memory more, but by changing what
we ask it to do. Instead of asking the model to *recall* a fact, we give
it the real, specific document and ask it to *read and report* the number
that's actually in front of it. This pattern — find the right document,
then have the model answer using it — is what people mean when they say
"RAG" (retrieval-augmented generation). By the end of this notebook you'll
have built a small, real, working version of one yourself.

## What you'll be able to answer by the end

1. If a model can't be trusted to recall a specific fact from memory, can
   it still answer correctly when the fact is put directly in front of it?
2. Where does the right document actually come from, if you don't already
   know which one you need?
3. Does giving the model a document guarantee a correct answer — or can it
   still go wrong?
4. What happens when nothing relevant exists to hand the model at all?
5. What does this actually change about how much you can trust an AI
   tool's answer at work?


## 1. What problem are we investigating?

In notebook 3, we asked the real model to finish this sentence:

> "Field notes: at Well A-12, the crew pressure tested the 9-5/8 inch
> casing string, held steady at ___"

There is no real Well A-12 — we made it up — so any number the model
produces there cannot be a real fact it looked up. It's just a
plausible-sounding guess dressed up as a fluent sentence.

Now imagine a different situation: Well A-12 **is** real, and your company
**does** have a real completion report for it sitting in a file somewhere,
with the real pressure test value written down. The model still hasn't
memorized that specific document — it was never trained on your company's
private files — so asking it the same way would still just produce a
guess. But what if, instead of asking the model to remember, we simply
**showed it the report** and asked it to read the number off the page?

That's the entire idea this notebook tests, live, with the real model.


## Installation (run once)

If you already set up the environment for notebooks 1-3, you don't need to
do anything further. Otherwise, uncomment and run the cell below once.


In [1]:
# Run this once. After the packages are installed you can leave this commented out.
# %pip install torch transformers accelerate pandas matplotlib numpy
print("If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.")


If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.


## 2. Load the real AI model

Same model, same computer setup as notebooks 1-3. As before, **you don't
need to understand the next code cell; just run it.**


In [2]:
import random
import sys

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PRIMARY_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
PRIMARY_MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"  # pinned for reproducibility
FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # used only if the primary model fails to load
FALLBACK_MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"  # pinned for reproducibility

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

DTYPE = torch.float16 if DEVICE in ("mps", "cuda") else torch.float32

print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Selected device:      {DEVICE}")
print(f"Selected dtype:       {DTYPE}")


Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Selected device:      mps
Selected dtype:       torch.float16


In [3]:
def load_model(model_name: str = PRIMARY_MODEL_NAME):
    '''Load a causal language model and its tokenizer onto the selected device.

    Falls back to FALLBACK_MODEL_NAME if the primary model cannot be loaded,
    and always reports which model actually ended up running.
    '''
    try:
        tok = AutoTokenizer.from_pretrained(model_name, revision=PRIMARY_MODEL_REVISION)
        mdl = AutoModelForCausalLM.from_pretrained(model_name, revision=PRIMARY_MODEL_REVISION, dtype=DTYPE, use_safetensors=True)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, model_name
    except Exception as exc:  # noqa: BLE001 - report and fall back, don't crash the notebook
        print(f"Could not load '{model_name}' ({exc}). Falling back to '{FALLBACK_MODEL_NAME}'.")
        tok = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME, revision=FALLBACK_MODEL_REVISION)
        mdl = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL_NAME, revision=FALLBACK_MODEL_REVISION, dtype=DTYPE, use_safetensors=True)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, FALLBACK_MODEL_NAME


tokenizer, model, MODEL_NAME = load_model()
print(f"\nModel actually loaded and used in this notebook: {MODEL_NAME}")



Model actually loaded and used in this notebook: Qwen/Qwen2.5-1.5B-Instruct


## 3. Reusing what earlier notebooks already built

The same generation function from notebooks 2 and 3: repeatedly ask the
model for its single top answer and add it to the text. We use this
"always play it safe" (greedy) rule throughout this notebook, for the same
reason as notebook 3 — it's fully predictable, which makes it the fairest
way to compare answers.


In [4]:
def get_next_token_distribution(prompt: str):
    '''Run `prompt` through the model and return (input_ids, logits, probabilities) for the next token.'''
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**enc)
    logits_vec = out.logits[0, -1, :]
    probs_vec = torch.softmax(logits_vec, dim=-1)
    return enc["input_ids"][0], logits_vec, probs_vec


def generate_greedy(prompt: str, n_new_tokens: int) -> str:
    '''Repeatedly take the model's single top answer and add it to the text.'''
    text = prompt
    for _ in range(n_new_tokens):
        _, logits_vec, _ = get_next_token_distribution(text)
        next_id = int(torch.argmax(logits_vec).item())
        text += tokenizer.decode([next_id])
    return text


def show(df: pd.DataFrame):
    '''Display a table without pandas' default row-number column on the left, which isn't real data.'''
    display(df.style.hide(axis="index"))


## 4. Recap: the ungrounded guess, live

First, let's reproduce notebook 3's result in this notebook, so everything
here is self-contained. Same prompt, same real model, same "always play it
safe" rule — so this should, and does, produce the exact same answer as
before.


In [5]:
UNGROUNDED_PROMPT = "Field notes: at Well A-12, the crew pressure tested the 9-5/8 inch casing string, held steady at"

ungrounded_result = generate_greedy(UNGROUNDED_PROMPT, n_new_tokens=8)
print(ungrounded_result)
print("(Well A-12 is fictional -- this number is not a real engineering spec.)")


Field notes: at Well A-12, the crew pressure tested the 9-5/8 inch casing string, held steady at 1,000 psi.
(Well A-12 is fictional -- this number is not a real engineering spec.)


**What this shows:** exactly what notebook 3 found — a fluent,
specific number, with nothing real behind it. The model was never given
anything to look at; it's reciting whatever pattern of words is most
common in similar-sounding sentences. Let's now change that.


## 5. The fix: show the model a real document

Here is a short, invented-for-this-notebook "completion report" for Well
A-12. Pretend this is a real file your company actually has on record.


In [6]:
DOCUMENT_A12 = (
    "Well A-12 completion report: the crew pressure tested the 9-5/8 inch "
    "casing string and held steady at 4,500 psi for 30 minutes, witnessed "
    "by the site supervisor."
)

print(DOCUMENT_A12)
print("(Invented for this notebook -- not a real document or a real spec.)")


Well A-12 completion report: the crew pressure tested the 9-5/8 inch casing string and held steady at 4,500 psi for 30 minutes, witnessed by the site supervisor.
(Invented for this notebook -- not a real document or a real spec.)


Now, instead of asking the model to recall a number from memory, we
build a prompt that puts the real document directly in front of it, and
asks it to answer using only what's written there.


In [7]:
GROUNDED_TEMPLATE = (
    "Reference document: {document}\n\n"
    "Question: What pressure did {well} hold during its casing pressure test?\n"
    "Answer:"
)

grounded_prompt_a12 = GROUNDED_TEMPLATE.format(document=DOCUMENT_A12, well="Well A-12")
grounded_result_a12 = generate_greedy(grounded_prompt_a12, n_new_tokens=8)

print(grounded_result_a12[len(grounded_prompt_a12):].strip())


4,500 psi


**What this shows — read the real output above.** With the document
placed directly in the prompt, the model's answer matches the number
actually written in the document, not the guess it produced in Section 4.
Nothing about the model changed. What changed is what text it was reading
from the moment it started predicting. This is the entire mechanical idea
behind "grounding" an AI tool's answers: put the real source text in the
prompt, and let the model's next-token prediction — the same mechanism
from notebook 1 — do the reading and summarizing, instead of the
recalling.


## 6. Where does the right document come from?

Section 5 worked because a human (us) already knew which document was
relevant and pasted it in by hand. In real use, you don't want to have to
find and paste in the right file yourself every time — you want the
system to find it for you out of everything you have on file. That
automatic "find the right document first" step is what the **R** in RAG
(retrieval-augmented generation) refers to: **retrieval**.

Let's build a small, real example: a handful of completion reports for
different wells, and a way to automatically pick out the one that
actually matches a question.


In [8]:
document_store = {
    "Well A-12": (
        "Well A-12 completion report: the crew pressure tested the 9-5/8 "
        "inch casing string and held steady at 4,500 psi for 30 minutes, "
        "witnessed by the site supervisor."
    ),
    "Well B-7": (
        "Well B-7 completion report: the crew pressure tested the 9-5/8 "
        "inch casing string and held steady at 3,200 psi for 30 minutes, "
        "witnessed by the site supervisor."
    ),
    "the North Pad well": (
        "North Pad well completion report: the crew pressure tested the "
        "9-5/8 inch casing string and held steady at 5,000 psi for 45 "
        "minutes, witnessed by the site supervisor."
    ),
    "Well 204": (
        "Well 204 completion report: the crew pressure tested the 7 inch "
        "liner and held steady at 2,800 psi for 20 minutes, witnessed by "
        "the site supervisor."
    ),
    "the offshore platform well": (
        "Offshore platform well completion report: the crew pressure "
        "tested the 9-5/8 inch casing string and held steady at 6,000 psi "
        "for 60 minutes, witnessed by the regulatory inspector."
    ),
}

for well, doc in document_store.items():
    print(f"{well}: {doc}")
    print()


Well A-12: Well A-12 completion report: the crew pressure tested the 9-5/8 inch casing string and held steady at 4,500 psi for 30 minutes, witnessed by the site supervisor.

Well B-7: Well B-7 completion report: the crew pressure tested the 9-5/8 inch casing string and held steady at 3,200 psi for 30 minutes, witnessed by the site supervisor.

the North Pad well: North Pad well completion report: the crew pressure tested the 9-5/8 inch casing string and held steady at 5,000 psi for 45 minutes, witnessed by the site supervisor.

Well 204: Well 204 completion report: the crew pressure tested the 7 inch liner and held steady at 2,800 psi for 20 minutes, witnessed by the site supervisor.

the offshore platform well: Offshore platform well completion report: the crew pressure tested the 9-5/8 inch casing string and held steady at 6,000 psi for 60 minutes, witnessed by the regulatory inspector.



Five wells, five reports, five different real numbers written into
them on purpose — so we can check, honestly, whether the right one gets
picked out for a given question. (Notice "Well C-3" from notebook 3's
well list is deliberately left out of this document store — we'll come
back to that gap in Section 10.)


## 7. A simple, real, fully visible way to find the right document

Production RAG systems usually find the right document using "embeddings"
— turning text into lists of numbers and comparing them for closeness in
meaning. (The advanced, more technical part of this project tests that
approach directly on this same model, and is honest about where it does
and doesn't actually work for oilfield language.)

For this notebook, we'll use something simpler and completely visible:
**count how many of the same important words appear in the question and
in each document.** The document that shares the most words with the
question is our best guess at the right one. It's not as capable as a
real meaning-based search, but every step of it can be seen and checked —
which is exactly the point of this notebook.


In [9]:
import re

STOPWORDS = {
    "a", "an", "and", "at", "by", "did", "during", "for", "hold", "holding",
    "how", "in", "is", "it", "its", "of", "on", "test", "tested", "testing",
    "the", "to", "was", "were", "what", "which", "with",
}


def keyword_overlap(query: str, document: str):
    '''Return (score, shared_words) counting distinct important words shared between query and document.'''
    query_words = {w for w in re.findall(r"[a-z0-9\-]+", query.lower()) if w not in STOPWORDS}
    doc_words = {w for w in re.findall(r"[a-z0-9\-]+", document.lower()) if w not in STOPWORDS}
    shared = sorted(query_words & doc_words)
    return len(shared), shared


question = "What pressure did Well A-12 hold during its casing pressure test?"

rows = []
for well, doc in document_store.items():
    score, shared = keyword_overlap(question, doc)
    rows.append({"Document": well, "Shared words": ", ".join(shared), "Match score": score})

retrieval_table = pd.DataFrame(rows).sort_values("Match score", ascending=False)
show(retrieval_table)


Document,Shared words,Match score
Well A-12,"a-12, casing, pressure, well",4
Well B-7,"casing, pressure, well",3
the North Pad well,"casing, pressure, well",3
the offshore platform well,"casing, pressure, well",3
Well 204,"pressure, well",2


**What this shows — read the real table above.** Every document
shares a few generic oilfield words with the question ("pressure",
"casing"), but only the Well A-12 document also shares the words that
actually identify *which* well the question is about ("well", "a-12").
That's enough for it to score highest and come out on top — using nothing
but a real, visible word count, no hidden model involved in this step at
all.


## 8. Put it together: retrieve, then answer

Now let's wire the two real pieces together into one function: find the
best-matching document automatically (Section 7), then hand it to the
model as grounding (Section 5). This is a small, real, working
retrieval-augmented generation pipeline.


In [10]:
def retrieve(query: str, store: dict):
    '''Return the (well, document, score) with the highest keyword-overlap score in store.'''
    best_well, best_doc, best_score = None, None, -1
    for well, doc in store.items():
        score, _ = keyword_overlap(query, doc)
        if score > best_score:
            best_well, best_doc, best_score = well, doc, score
    return best_well, best_doc, best_score


def answer_with_rag(query: str, well: str, store: dict = document_store, n_new_tokens: int = 8):
    '''Retrieve the best-matching document for `well`, then generate a grounded answer.'''
    retrieved_well, retrieved_doc, score = retrieve(query, store)
    prompt = GROUNDED_TEMPLATE.format(document=retrieved_doc, well=well)
    full_result = generate_greedy(prompt, n_new_tokens=n_new_tokens)
    answer = full_result[len(prompt):]
    return retrieved_well, score, answer


for well, query in [
    ("Well A-12", "What pressure did Well A-12 hold during its casing pressure test?"),
    ("Well B-7", "What pressure did Well B-7 hold during its casing pressure test?"),
    ("the North Pad well", "What pressure did the North Pad well hold during its casing pressure test?"),
]:
    retrieved_well, score, answer = answer_with_rag(query, well)
    print(f"Asked about:      {well}")
    print(f"Document retrieved: {retrieved_well} (match score {score})")
    print(f"Model's answer:     {answer.strip()}")
    print()


Asked about:      Well A-12
Document retrieved: Well A-12 (match score 4)
Model's answer:     4,500 psi



Asked about:      Well B-7
Document retrieved: Well B-7 (match score 4)
Model's answer:     3,200 psi



Asked about:      the North Pad well
Document retrieved: the North Pad well (match score 5)
Model's answer:     5,000 psi



**What this shows — read the real output above.** For all three
wells, the retrieval step picked out that well's own document, and the
model's answer matches that document's real number. This isn't limited to
the one example from Section 5 — the same two-step pipeline (find the
right document, then read the answer off it) generalizes across different
wells and different questions, because both steps are doing something
real rather than memorized.


## 9. What if retrieval finds the wrong document?

Section 8 worked because the retrieval step found the *correct* document
every time. But retrieval is just a search — it can be wrong. What
actually happens if the model is handed the *wrong* document by mistake?
Let's force that, on purpose, and watch honestly.


In [11]:
wrong_prompt = GROUNDED_TEMPLATE.format(document=document_store["Well B-7"], well="Well A-12")
wrong_result = generate_greedy(wrong_prompt, n_new_tokens=8)

print(wrong_result[len(wrong_prompt):].strip())


3,200 psi


**What this shows — read the real output above.** Handed Well
B-7's report while being asked about Well A-12, the model answers with
Well B-7's number anyway. It has no way to notice the mismatch — it isn't
checking the document against some separate, true memory of Well A-12; it
is simply reading whatever text sits in front of it and reporting the
number that's there. This is the single most important limit of grounding
to understand: **a wrong or irrelevant document produces a wrong answer
just as fluently and confidently as a correct one.** Grounding only helps
if the retrieval step actually finds the right source — get that step
wrong, and you now have a confidently wrong answer that *looks* like it
came with a citation.


## 10. What if nothing relevant exists at all?

Section 6 deliberately left "Well C-3" out of the document store — there
is no real report for it anywhere in our five documents. In real use,
this happens constantly: someone asks about something that genuinely
isn't in your files yet. What should happen is the system telling you
that, honestly, rather than guessing. Let's test, with the real model,
whether it actually does.

First, the plain grounded template from Section 5, with whatever document
retrieval finds for a well that isn't really there:


In [12]:
c3_query = "What pressure did Well C-3 hold during its casing pressure test?"
retrieved_well, score, plain_answer = answer_with_rag(c3_query, "Well C-3")

print(f"Document retrieved for 'Well C-3': {retrieved_well} (match score {score})")
print(f"Model's answer: {plain_answer.strip()}")


Document retrieved for 'Well C-3': Well A-12 (match score 3)
Model's answer: 4,500 psi for


Now let's try a second version of the template that explicitly
tells the model what to do when the answer isn't really there, and see
whether that changes anything:


In [13]:
HONEST_TEMPLATE = (
    "Reference document: {document}\n\n"
    "Question: What pressure did {well} hold during its casing pressure test?\n"
    "If the document above does not contain this information, respond with "
    "exactly: Not found in the provided document.\n"
    "Answer:"
)

_, retrieved_doc_c3, score_c3 = retrieve(c3_query, document_store)
honest_prompt = HONEST_TEMPLATE.format(document=retrieved_doc_c3, well="Well C-3")
honest_result = generate_greedy(honest_prompt, n_new_tokens=12)

print(f"Document retrieved for 'Well C-3': {retrieved_well} (match score {score_c3})")
print(f"Model's answer: {honest_result[len(honest_prompt):].strip()}")


Document retrieved for 'Well C-3': Well A-12 (match score 3)
Model's answer: Not found in the provided document. Based on the information given


**What this shows — read the real outputs above, both of them.**
This is the honest, unscripted result of asking about something that
genuinely isn't in the document store, tested two different ways. Notice
that the match score here is real but low — retrieval still hands back
*some* document (Well A-12's, as it happens, sharing only generic words
like "casing" and "pressure"), because nothing in our simple
word-counting method knows how to say "none of these are actually
relevant."

The two answers show a real difference. The plain template confidently
answers **"4,500 psi"** — Well A-12's number, presented as if it were Well
C-3's, with no hint anything is wrong. The template that explicitly
instructed the model to admit when information is missing actually
opens with **"Not found in the provided document."** — it worked, this
time — before drifting into extra, unrequested text afterward, a reminder
that we only asked the model to stop after a fixed number of words rather
than telling it to stop once it had answered. Adding the instruction
measurably helped here; it did not make the output perfectly clean.


## 11. What this means: grounding helps, but it isn't magic

Put Sections 4 through 10 together, and the honest picture looks like
this:

- **Grounding works, and we just proved it, live.** When the real
  document is in front of the model, its answer matches that document —
  a real, checkable improvement over notebook 3's ungrounded guessing.
- **Retrieval is now the critical step.** The model doesn't add any
  independent judgment about whether the document it was handed is the
  right one — Section 9 showed it will confidently repeat a wrong
  document's number just as fluently as a right one.
- **"Nothing relevant found" has to be handled on purpose.** A search
  step that always returns *something*, even when nothing truly matches,
  can quietly hand the model an irrelevant document — Section 10 showed
  exactly what our simple retrieval method does in that situation, with
  real, honestly reported results either way.
- **The real practical win isn't "the model can't be wrong anymore."**
  It's that a grounded answer now comes with something notebook 3's
  answers never had: a specific, real, attached document you or a
  colleague can actually go check. That's the difference between a claim
  you have to take on faith and one you can verify in seconds.


## 12. Key lessons

1. Putting the real source document directly in the prompt changes the
   model from *guessing* to *reading* — we proved this with a real,
   working example in Section 5.
2. Retrieval — automatically finding the right document — can be done
   with something as simple and fully visible as counting shared words,
   and it worked correctly across multiple wells in Section 8.
3. A wrong or irrelevant document produces a wrong answer just as
   fluently as a correct document produces a right one (Section 9). The
   model cannot tell the difference on its own.
4. When nothing relevant actually exists, a real retrieval system can
   still hand back *something* — whether the model then admits it doesn't
   know, or answers anyway, has to be checked, not assumed (Section 10).
5. Grounding's real value isn't a guarantee of correctness — it's that a
   grounded answer comes with a real, specific source a human can verify,
   which an ungrounded guess never has.


## 13. Try your own example

Write your own short "document" about something from your own work, and
your own question about it. Change the two lines below and run the cell.


In [14]:
my_document = (
    "Torque specification sheet: the connection shall be made up to a "
    "final torque of 100 Nm, verified with a calibrated torque wrench."
)  # <-- change this line
my_question = "What is the final make-up torque specified?"  # <-- change this line

my_prompt = f"Reference document: {my_document}\n\nQuestion: {my_question}\nAnswer:"
my_result = generate_greedy(my_prompt, n_new_tokens=16)
print(my_result[len(my_prompt):].strip())


The final make-up torque specified is 100 Nm. 

This


## 14. Optional exercises

1. In Section 6, add a sixth document of your own (a new fictional well)
   and confirm in Section 7-8 that retrieval and grounding both still work
   correctly for it.
2. In Section 9, try handing the model a document that's only *partly*
   relevant (for example, correct well, wrong casing size) and see how it
   handles a partial mismatch.
3. In Section 10, try rewording the "if not found" instruction in
   `HONEST_TEMPLATE` and see whether a different phrasing changes the real
   result.
4. Try writing a query in Section 7 using completely different words than
   the document (a paraphrase). Does the simple word-counting retrieval
   still find the right document? What does that tell you about its
   limits compared to a real meaning-based search?
5. If you're curious how a real meaning-based ("embedding") search
   actually performs on oilfield language instead of simple word
   counting, see `advanced/03_embeddings_and_attention.ipynb` — it tests
   that directly, with real results, including where it falls short.


## 15. Technical appendix (optional — skip if you like)

**Why word-counting instead of embeddings for retrieval.** Real production
RAG systems typically use a dedicated embedding model to turn text into
vectors and rank documents by vector similarity (often cosine similarity).
That approach is more capable in general, but it's also a second model
whose behavior isn't visible step by step. This notebook deliberately uses
plain keyword overlap instead, so every part of the retrieval decision —
which words matched, and how many — is fully readable in Section 7's
table. `advanced/03_embeddings_and_attention.ipynb` covers the
embedding-based approach directly on this same model, including a real,
honestly-reported finding about where it does and doesn't capture oilfield
meaning well.

**Model and environment actually used in this run** (printed live, not
hard-coded):


In [15]:
print(f"Model:                {MODEL_NAME}")
print(f"Device:               {DEVICE}")
print(f"Dtype:                {DTYPE}")
print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Random seed:          {SEED}")


Model:                Qwen/Qwen2.5-1.5B-Instruct
Device:               mps
Dtype:                torch.float16
Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Random seed:          42


**A note on model size.** This notebook uses the same small (1.5B
parameter) model as the rest of the series, for the same reasons: it runs
locally, offline, on a normal laptop. The core lesson — that grounding
changes recall into reading, and that retrieval quality is what actually
determines whether that reading is trustworthy — applies to
retrieval-augmented systems generally, not to this model specifically.
